In [14]:
import pandas as pd 

x = pd.read_csv("diabetes_prediction_dataset.csv")

x.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


Este modelo utiliza Aprendizado de Máquina Supervisionado para estimar o nível de glicose no sangue (blood_glucose_level) a partir de variáveis clínicas e demográficas.

1. Importação de Bibliotecas

pandas: Essencial para carregar e manipular sua tabela (DataFrame).
train_test_split: Divide seus dados para que o modelo não "decore" as respostas (overfitting).
LabelEncoder: Transforma textos (ex: "Male", "Female") em números (0, 1), pois modelos de IA só processam matemática.
RandomForestRegressor: O "cérebro". É um conjunto de múltiplas árvores de decisão que votam para chegar a um resultado numérico.


2. Tratamento de Dados (Pré-processamento)

Modelos de regressão não aceitam strings. O LabelEncoder é usado nas colunas gender e smoking_history.
Dica para o futuro: Se houver valores "No Info", o LabelEncoder os tratará como uma categoria à parte. No futuro, você pode tentar remover essas linhas para ver se a precisão melhora.

3. Separação de X (Features) e y (Target)

X: São as "pistas" (Idade, IMC, Hipertensão, etc.). Removemos a Glicose daqui, senão o modelo estaria "colando" na prova.
y: É o que queremos descobrir (o valor da Glicose).

4. Divisão Treino/Teste

Usamos 80% dos dados para o modelo aprender os padrões e 20% para testá-lo. O random_state=42 garante que, se você rodar o código de novo, os resultados sejam os mesmos.

5. Treinamento e Avaliação

O mean_absolute_error (MAE) nos diz o quanto o modelo erra "para mais ou para menos". No seu caso, um erro de 12,28 mg/dL significa que se a glicose real é 100, o modelo previu algo entre 87,7 e 112,3.


In [4]:
# Bibliotecas: 

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [7]:
df = pd.read_csv("diabetes_prediction_dataset.csv")

# 2. Pré-processamento: Transformar texto em números
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['smoking_history'] = le.fit_transform(df['smoking_history'])

# 3. Definir X (pistas) e y (o que queremos prever: Glicose)
# Removemos a glicose de X e usamos como y
X = df.drop('blood_glucose_level', axis=1) 
y = df['blood_glucose_level']

# 4. Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Escolher o Modelo (Regressão porque o resultado é um número contínuo)
modelo = RandomForestRegressor(n_estimators=100, random_state=42)

# 6. Treinar o modelo
modelo.fit(X_train, y_train)

# 7. Testar
previsoes = modelo.predict(X_test)

# 8. Avaliar (Erro Médio Absoluto em mg/dL)
mae = mean_absolute_error(y_test, previsoes)
print(f"O modelo erra, em média, {mae:.2f} mg/dL na previsão da glicose.")

O modelo erra, em média, 32.28 mg/dL na previsão da glicose.


segundo teste com a bilbioteca do pyspark

In [2]:
!pip install pyspark

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# Cria a sessão Spark 

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GlicosePrediction") \
    .getOrCreate()

In [ ]:
# Transformar as colunas de texto em números usando StringIndexer

from pyspark.ml.feature import StringIndexer

indexer_gender = StringIndexer(inputCol="gender", outputCol="gender_index")
indexer_smoking = StringIndexer(inputCol="smoking_history", outputCol="smoking_index")

df = indexer_gender.fit(df).transform(df)
df = indexer_smoking.fit(df).transform(df)

In [ ]:
# Criar vetor de features (X)

from pyspark.ml.feature import VectorAssembler

features = [col for col in df.columns if col != "blood_glucose_level"]

assembler = VectorAssembler(inputCols=features, outputCol="features")

df = assembler.transform(df)
df.select("features", "blood_glucose_level").show()